PROGETTO 5: OBJECT DETECION AVANZATA E ANALISI DEL FLUSSO VIDEO

Passiamo dalla statitica alla dinamica.
La vita vera è un flusso costante un file che non si ferma mai.

Non guardiamo più cosa c'è nel frame ma dove sta andando.

- Come trasformare una linea di pixel in un sensore intelligente
- ottimizzazone real-time
- come trasformare dati grezzi in insight statistici.

Come creiamo un confine invisibile?

Varchi Virtuali e Logica di Conteggio
In contesti industriale, o di sorveglianza urbana, c'è spesso la necesssità di quantificare il flusso tramite varchi. Contare quante auto entrano nella zona ZTL richiede una logica di transito.
Nel Deep Leearning un varco è un'idea matematica, è una linea gemoetrica, definita nel piano dell'immagine, che funge da trigger per l'incremento di un contatore. Ci interessa il momento esatto in cui un oggetto attraversa il confine della linea geometrica.

Ma come facciamo a non contare 10 volte la stessa persona, lo stesso oggetto, che cammina avanti e indietro tra il varco?
Qui entra il gioco il Centroid Tracking: riduci ogni oggetto ad un singolo punto luminoso, il suo baricentro, monitorando questo punto, crei una scia. Se la scia è a sinistra del varco e dopo è a destra del varco, abbiamo l'evento (un passaggio) e l'incremento del contatore. 
E' fondamentale che ad ogni oggetto sia associato un ID Persistence (un numero), un identificatore univoco per garantire che il conteggio avvenga una sola volta.

Il mondo reale però è sporco e le nostra bouding box non sono sempre stabili

Dettagli Implementativi del Varco
- Gestione del rumore posizoinale: la bounding box possono oscillare leggermente a causa dell'incertezza del modello (rumore del modello). Implementare un buffer o una zona di tolleranza attorna alla linea previene falsi conteggi dovuti a micro-movimento. Se la linea virtuale è sottilissima quel tremolio potrebbe far scattare il contatore continuamente (avanti e indietro). Per risolvere usiamo dei buffer o zone di tolleranza, come dare al varco uno spessore, una terra di nessuno, un oggetto deve uscire  ed entrare in questa zona aumentata per aumentare il contatore.
- Direzionalità del flusso: monitorando il segno del prodotto scalare tra il vettore di movimento e la normale linea, possiamo distinguere tra entrate e uscite, utile per monitorare varchi bidirezionali
- Multi-line support: in scenari complessi è possibile definire più linee per monitorare diverse corsie o aree di interesse, assegnando a ciascuna un contatore indipendente nel sistema.

Ma come traduciamo tutto questo in logica booleana?

Matematica dell'Attraversamento
Equazione della retta e posizione del punto
Per detarminare se un punto ha attraversato una linea, analizziamo il segno della funzione della retta passante per due punti fissi. Se il segno cambia tra il frame attuale e il precedente, l'attraversamento è avvenuto.
Chiediamo al computer se il centroide si trova sopra o sotto la linea, la magia avviene quando confrontiamo questa posizione in due tempi diversi.
Siano P1 e P2 i punti che definiscono il varco, La posizoine del centroide C rispetto alla linea si ottiene tramite il prodotto vettoriale o l'equazione implicita della retta.

Questione della velocità: perchè il video non aspetta nessuno

Ottimizzazione del Tempo Reale.
Tecniche per mantenere elevati FPS senza sacrificare la precisione
L'Object detection è computazionalmente costosa. In un flusso video a 30 FPS, elaborare ogni singolo frame con modelli pesanti come YOLOv8 o SSD può causare colli di bottiglia e latenza inaccettabili nel feedback visivo (la CPU di scalda ed il video diventa una sequenza di scatti fastidiosi).
Ottimizzare significa implementare strategie di 'inference Skipping' e sfruttare l'accelerazione hardware. Vediamo come delegare parte del lavoro a algoritmi di tracking più leggeri tra un'inferenza neurale e l'altra.

Quali sono le tecniche per essere veloci senza diventare imprecisi?

Strategia di Efficienza
Riduzione del carico computazionale sulla CPU e GPU
- Frame Skipping: eseguire l'inferenza del modello Deep Learning solo ogni N frame (es. ogni 3 o 5 frame). Usiamo i Deep Learning ogni 3 o 5 frame, nei frame intermedi usiamo la Kalman Filtering
- Kalman Filtering: predire la posizioe futura dell'oggetto basandosi sulla velocità riducendo la necessità di rilevamenti continui
- Resolution Scaling: processare l'immagine a una risoluzione ridotta per accelerare i tempi di convoluzione del modello
- Async Processing: utilizzare thread separati per la decodifica del video, l'inferenza e la visualizzazione a schermo.

Ma se volessi spingere ancora di più sull'accelleratore?

Integrazione Tracking-Detection
- DeepSORT e ByteTrack: l'uso di tracker moderni permette di mantenere l'identità dell'oggetto anche in caso di occlusioni temporanee, delegando la ri-identificazione a feature vettoriali estratte dai volto o dalle silhouette.
- Ottimizzazione Hardware: l'impiego di librerie come TensorRT o OpenVINO permette di convertire i pesi del modelli in formati ottimizzati per specifiche architetture GPU o NPU, migliorando drasticamente il throughput
- Batching di inferenza: se il sistema gestisce più telecamere, raggruppare i frame in un unico batch per l'inferenza parallela ottimizza l'uso della memoria video e dei cicli di clock della GPU.

Ma c'è un limite fisico che non possiamo ignorare (il tempo di campionamento)

Analisi della Latenza
Il trade-off tra velocità e accuratezza
Il tempo totale di elaborazione per frame deve essere inferiore al tempo di campionamento del video. Se elaboriamo a 30 FPS, abbiamo circa 33ms per completare tutte le operazioni, incluso il disegno a schermo.
La latenza totale L è data dalla somma dei tempi di acquisizione, pre-processing, inferenza e post-processing. L'obbiettivo è minimizzare questa somma senza degradare eccessivamente la mAP

Reportistica e Statistiche
Estrarre valore dai dati grezzi di rilevamento
Dal monitoraggio all'analisi, passando dalla domanda cosa sta succendo ora? alla domanda cosa è successo nelle ultime 24 ore?
Il conteggio in tempo reale è utile per il monitoraggio immediato, ma il valore a lungo termine risiede nei dati storici. Trasformare gli eventi di attraversamento in dataset strutturati permette analisi di business e previsioni di traffico.
Vediamo come strutturare un log di eventi che registri timestamp, classe dell'oggetto, direzione e confidenza del rilevamento, facilitando la creazione di dashboard e report periodici.

Architettura del Dato
Dalla RAM alla persistenza del dato
L'archtettura del dato deve essere leggere, non scriviamo sul disco ogni singolo movimento, questo perchè I/O è lento.
- Event Logging: registrazione immediata dell'attraversamento in una struttura di dati in-memory per miniimzzare l'I/O blocking
- Data Aggregation: raggruppamento dei conteggi per fasce orarie (per es. ogni 15 minuti) per identificare i picchi di affluenza. Non mi interessa sapere che l'auto con ID 65 è passata alle 15.02, mi intessa sapere che tra le 15.00 e le 15.15 sono passate tot auto.
- Export Multi-formato: generazione automatica di file CSV o JSON per l'integrazione con strumenti di Business Intelligence esterna. Trasformiamo i pixel in grafici comprensibili
- Visualizzazione Dashboard: creazione di grafi a barre o line char per rappresentare visivamente l'andamento del flusso nel tempo.

Ma possiamo visualizzare i dati in modo più creativo?

Metriche di Performance del Flusso
- Heatmap di traiettoria: oltre al conteggio, visualizzare dove gli oggetti passano più frequentemente aiuta a ottimizzare la disposizione degli spazi o dei varchi fisici in un ambiente reale.
- Analisi dei tempi di sosta: calcolando il tempo intercorso tra l'ingresso in una area ROI e l'attraversamento del varco, possiamo dedurre la velocità media del fusso o eventuali ingorghi.
In un negozio possiamo capire se la vetrina sta attirando l'attenzione o se la gente passa senza guardare.
- Itegrazione con database: per progetti su larga scala, l'invio asincrono dei dati a un database SQL o NoSQL permette la centralizzazione dei log provenienti da molteplici nodi di visione periferica.

Cè poi un'ultima analisi per rendere i dati leggibili

Analisi Statistica del Traffico
Media Mobile e Smoothing
I dati di conteggio grezzi possono essere molto volatili. Analizzare i dati minuto per minuto rende il dato caotico e con estremi picchi. Applicare una media mobile permette di filtrare le fluttuazioni casuali e identificare i trend reali del movimento.
La media mobile semplice (SMA) su un intervallo di k periodi aiuta a stabilizzare la curva di reportistica per una lettura più chiara degli insight

In [1]:
import cv2
import datetime
from ultralytics import YOLO

# ==============================================================================
# CONFIGURAZIONE DEL SISTEMA
# ==============================================================================
# Questi parametri definiscono il comportamento dell'algoritmo di visione artificiale.
MODELLO = "yolov8m.pt"  # Carica i pesi di YOLOv8 versione Medium (bilanciamento tra velocità e precisione)
LINEA_Y = 350           # Coordinata Y della "linea virtuale" di traguardo
OFFSET = 15             # Area di tolleranza (pixel) intorno alla linea per catturare il movimento
FILE_LOG = "log_accessi.csv" # Nome del database testuale dove salvare i passaggi

# ==============================================================================
# INIZIALIZZAZIONE COMPONENTI
# ==============================================================================

# 1. Caricamento del Modello YOLO (You Only Look Once)
# La classe YOLO gestisce il caricamento della rete neurale e fornisce i metodi per l'inferenza.
model = YOLO(MODELLO)

# 2. Configurazione Sorgente Video
# cv2.VideoCapture(0) accede alla webcam predefinita. Crea un oggetto 'cap' che gestisce lo streaming.
cap = cv2.VideoCapture(0)

# 3. Stato del Sistema
conta_totale = 0          # Contatore globale dei passaggi rilevati
id_gia_contati = set()    # Set (insieme unico) per memorizzare gli ID univoci già passati (evita doppi conteggi)

# 4. Preparazione File di Log
# Apriamo il file in modalità 'append' (a) per non sovrascrivere dati precedenti e scriviamo l'intestazione.
with open(FILE_LOG, "a") as f:
    f.write("Timestamp,Classe,ID,Totale\n")

print(f"[INFO] Sistema pronto. Modello: {MODELLO}. Tracker: Kalman attivo. Premi 'q' per uscire.")

# ==============================================================================
# LOOP PRINCIPALE DI ELABORAZIONE (Real-Time)
# ==============================================================================
while True:
    # A. Cattura del Frame
    # 'ret' è un booleano (True se il frame è catturato), 'frame' è la matrice dell'immagine (NumPy array).
    ret, frame = cap.read()
    if not ret: 
        break

    # B. Pre-Processing
    # Ridimensioniamo il frame a una dimensione standard per garantire fluidità (FPS) costante.
    frame = cv2.resize(frame, (1020, 720))
    
    # C. TRACCIAMENTO INTELLIGENTE (CORE LOGIC)
    # model.track() fa due cose in un colpo solo:
    # 1. Detection: Trova gli oggetti nel frame corrente.
    # 2. Tracking: Usa ByteTrack (con Filtro di Kalman) per associare gli oggetti ai frame precedenti.
    # persist=True: Fondamentale! Dice al modello di ricordare gli ID assegnati precedentemente.
    results = model.track(frame, persist=True, tracker="bytetrack.yaml", verbose=False, conf=0.4)

    # D. ANALISI DEI RISULTATI DEL TRACKER
    # Verifichiamo se il tracker ha assegnato almeno un ID (se ci sono oggetti tracciati).
    if results[0].boxes.id is not None:
        # Estraiamo i dati tecnici dai risultati di YOLO:
        # xyxy: Coordinate dei rettangoli (top-left, bottom-right).
        # id: Numeri identificativi univoci assegnati dal tracker ByteTrack.
        # cls: Classi numeriche (es. 0 per persona, 2 per auto).
        boxes = results[0].boxes.xyxy.cpu().numpy()
        track_ids = results[0].boxes.id.int().cpu().tolist()
        class_ids = results[0].boxes.cls.int().cpu().tolist()

        # Ciclo su ogni oggetto rilevato nel frame attuale
        for box, track_id, cls_id in zip(boxes, track_ids, class_ids):
            # 1. Calcolo del Centroide (Punto centrale dell'oggetto)
            x1, y1, x2, y2 = map(int, box)
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
            
            # Recuperiamo il nome leggibile della classe dal dizionario del modello
            nome_oggetto = model.names[cls_id]

            # 2. Rendering Visivo degli Oggetti
            # Cambia colore: Verde se già contato, Rosso se è un nuovo rilevamento.
            colore = (0, 255, 0) if track_id in id_gia_contati else (0, 0, 255)
            
            # Disegna il rettangolo attorno all'oggetto
            cv2.rectangle(frame, (x1, y1), (x2, y2), colore, 2)
            # Scrive ID e Nome sopra l'oggetto
            label = f"ID:{track_id} {nome_oggetto}"
            cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, colore, 2)
            # Disegna il punto centrale (centroide) che useremo per il conteggio
            cv2.circle(frame, (cx, cy), 4, (255, 0, 255), -1)

            # 3. LOGICA DI CONTEGGIO (VIRTUAL BOUNDARY)
            # Verifichiamo se il centroide (cy) si trova all'interno della "fascia di rilevamento" definita da LINEA_Y +- OFFSET.
            if (LINEA_Y - OFFSET) < cy < (LINEA_Y + OFFSET):
                # Se l'ID non è mai stato contato prima (grazie al Set id_gia_contati):
                if track_id not in id_gia_contati:
                    conta_totale += 1            # Incrementa contatore
                    id_gia_contati.add(track_id) # Registra l'ID per non contarlo più
                    
                    # Logica di salvataggio dati persistente
                    ora = datetime.datetime.now().strftime("%H:%M:%S")
                    with open(FILE_LOG, "a") as f:
                        f.write(f"{ora},{nome_oggetto},{track_id},{conta_totale}\n")
                    
                    # Segnale visivo di avvenuto conteggio: la linea flasha in bianco
                    cv2.line(frame, (0, LINEA_Y), (1020, LINEA_Y), (255, 255, 255), 5)

    # E. INTERFACCIA UTENTE (HUD - Heads-Up Display)
    # Disegna la linea di varco gialla (sempre visibile)
    cv2.line(frame, (0, LINEA_Y), (1020, LINEA_Y), (0, 255, 255), 2)
    
    # Crea un rettangolo nero semitrasparente nell'angolo in alto a sinistra per le statistiche
    cv2.rectangle(frame, (0, 0), (280, 60), (0, 0, 0), -1)
    cv2.putText(frame, f"PASSAGGI: {conta_totale}", (20, 40), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    # Mostra il frame elaborato in una finestra
    cv2.imshow("Sistema Visione IA - YOLOv8 + Kalman", frame)

    # F. GESTIONE USCITA
    # Aspetta 1ms l'input della tastiera. Se viene premuto 'q', esce dal loop.
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# G. PULIZIA RISORSE
# Rilascia la webcam e chiude tutte le finestre per liberare memoria.
cap.release()
cv2.destroyAllWindows()

[INFO] Sistema pronto. Modello: yolov8m.pt. Tracker: Kalman attivo. Premi 'q' per uscire.
